In [ ]:
# viscous part by spectural radius approximation

def compute_viscous_spectral_radius(mu, Pr, gamma, rho, xi, xj, mu_t=0.0, Pr_t=0.9):
    """
    Compute the viscous flux Jacobian approximation (spectral radius form).
    
    Based on equation (2.62) from Roger's
    |λ_max_ij| = (1/l_ij) * max(4/(3*rho_ij), gamma/rho_ij) * (mu*l_ij/Pr_l + mu_t_ij/Pr_t)
    
    Parameters
    ----------
    mu    : float  - dynamic viscosity (laminar), μ
    Pr    : float  - laminar Prandtl number, Pr_l
    gamma : float  - specific heat coefficient, γ
    rho   : float  - density, ρ_ij (at the edge midpoint or averaged)
    xi    : array-like (2D or 3D) - position vector of node i
    xj    : array-like (2D or 3D) - position vector of node j
    mu_t  : float  - turbulent dynamic viscosity, μ_t (default 0, laminar)
    Pr_t  : float  - turbulent Prandtl number, Pr_t (default 0.9)
    
    Returns
    -------
    lambda_max : float  - spectral radius |λ_max_ij|
    J_G        : ndarray - approximated Jacobian matrix (scalar * Identity)
    """
    xi = np.asarray(xi, dtype=float)
    xj = np.asarray(xj, dtype=float)

    # Edge vector length: l_ij = |x_j - x_i|
    l_ij = np.linalg.norm(xj - xi)
    if l_ij == 0.0:
        raise ValueError("Nodes xi and xj are coincident (l_ij = 0).")

    # Viscous scaling terms
    term1 = 4.0 / (3.0 * rho)   # 4 / (3 * rho_ij)
    term2 = gamma / rho          # gamma / rho_ij

    max_term = max(term1, term2)

    # Prandtl-weighted viscosity sum
    prandtl_term = (mu / Pr) + (mu_t / Pr_t)

    # Spectral radius (eq. 2.62)
    lambda_max = (1.0 / l_ij) * max_term * prandtl_term

    # Approximated Jacobian J_G = |lambda_max| * I  (eq. 2.61)
    n = len(xi)
    J_G = lambda_max * np.eye(n)

    return lambda_max, J_G

In [ ]:
# inviscid part by central-averaging of Ui and Uj
from sympy import *

def build_inviscid_jacobian():
    # ── Symbolic variables ────────────────────────────────────────────────────
    Ui  = Matrix(list(symbols("ui0:5")))   # shape (5,1) — now supports / and slicing
    Uj  = Matrix(list(symbols("uj0:5")))
    Ax, Ay, Az = symbols("Ax Ay Az")
    Aij = Matrix([Ax, Ay, Az])
    Vi  = symbols("V_i")
    g   = symbols("gamma")

    def dot(a, b):
        return (a.T @ b)[0, 0]

    # ── Primitive variables ───────────────────────────────────────────────────
    rhoi = Ui[0]
    ui   = Ui[1:4, :] / rhoi          # (3,1) Matrix / Symbol ✓
    rhoj = Uj[0]
    uj   = Uj[1:4, :] / rhoj

    # ── Ideal gas pressure ────────────────────────────────────────────────────
    pi = (Ui[4] - rhoi * dot(ui, ui) / 2) * (g - 1)
    pj = (Uj[4] - rhoj * dot(uj, uj) / 2) * (g - 1)

    # ── Central flux RHS ─────────────────────────────────────────────────────
    C = dot(rhoi * ui + rhoj * uj, Aij) / 2
    M = (rhoi * ui * ui.T + rhoj * uj * uj.T) @ Aij / 2
    G = (pi + pj) / 2 * Aij
    K = dot(((Ui[4] + pi) * ui + (Uj[4] + pj) * uj) / 2, Aij)

    Ri = -Matrix([[C], M + G, [K]]) / Vi

    # ── Symbolic Jacobian ─────────────────────────────────────────────────────
    print("Computing symbolic Jacobian (this may take a moment)...")
    all_vars    = Matrix([*Ui, *Uj])
    J_sym       = simplify(Ri.jacobian(all_vars))

    dRi_dUi_sym = J_sym[:, :5]
    dRi_dUj_sym = J_sym[:, 5:]

    # ── Lambdify ──────────────────────────────────────────────────────────────
    sym_args = (*Ui, *Uj, Ax, Ay, Az, Vi, g)   # 15 scalar arguments

    dRi_dUi_fn = lambdify(sym_args, dRi_dUi_sym, modules="numpy")
    dRi_dUj_fn = lambdify(sym_args, dRi_dUj_sym, modules="numpy")

    def jacobian_fn(Ui_val, Uj_val, Aij_val, Vi_val, gamma_val):
        import numpy as np
        Ui_val  = np.asarray(Ui_val,  dtype=float)
        Uj_val  = np.asarray(Uj_val,  dtype=float)
        Aij_val = np.asarray(Aij_val, dtype=float)

        args = (*Ui_val, *Uj_val, *Aij_val, float(Vi_val), float(gamma_val))

        dRi_dUi = np.array(dRi_dUi_fn(*args), dtype=float)
        dRi_dUj = np.array(dRi_dUj_fn(*args), dtype=float)

        return dRi_dUi, dRi_dUj

    return jacobian_fn


# ── Build once ────────────────────────────────────────────────────────────────
compute_inviscid_jacobian = build_inviscid_jacobian()


# ── Example usage ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import numpy as np

    gamma = 1.4
    rho_i, rho_j = 1.0, 1.1

    Ui = np.array([rho_i, rho_i*0.5, rho_i*0.2, 0.0, rho_i*2.5])
    Uj = np.array([rho_j, rho_j*0.4, rho_j*0.1, 0.0, rho_j*2.3])

    Aij   = np.array([1.0, 0.0, 0.0])
    Vi    = 0.1

    dRi_dUi, dRi_dUj = compute_inviscid_jacobian(Ui, Uj, Aij, Vi, gamma)

    np.set_printoptions(precision=4, suppress=True, linewidth=120)
    print("\ndRi/dUi (5x5):\n", dRi_dUi)
    print("\ndRi/dUj (5x5):\n", dRi_dUj)

In [ ]:
# roe-averaged inviscid jacobian
def build_inviscid_jacobian_roe():
    # ── Symbolic variables ────────────────────────────────────────────────────
    Ui  = Matrix(list(symbols("ui0:5")))
    Uj  = Matrix(list(symbols("uj0:5")))
    Ax, Ay, Az = symbols("Ax Ay Az")
    Aij = Matrix([Ax, Ay, Az])
    Vi  = symbols("V_i")
    g   = symbols("gamma")

    def dot(a, b):
        return (a.T @ b)[0, 0]

    # ── Primitive variables ───────────────────────────────────────────────────
    rhoi = Ui[0]
    ui   = Ui[1:4, :] / rhoi
    rhoj = Uj[0]
    uj   = Uj[1:4, :] / rhoj

    # ── Ideal gas pressure ────────────────────────────────────────────────────
    pi = (Ui[4] - rhoi * dot(ui, ui) / 2) * (g - 1)
    pj = (Uj[4] - rhoj * dot(uj, uj) / 2) * (g - 1)

    # ── Roe averaging ─────────────────────────────────────────────────────────
    R       = sqrt(rhoj / rhoi)
    rho_roe = sqrt(rhoi * rhoj)
    u_roe   = (ui + R * uj) / (1 + R)

    Hi      = (Ui[4] + pi) / rhoi
    Hj      = (Uj[4] + pj) / rhoj
    H_roe   = (Hi + R * Hj) / (1 + R)

    p_roe   = (g - 1) / g * rho_roe * (H_roe - dot(u_roe, u_roe) / 2)

    # ── Roe fluxes ────────────────────────────────────────────────────────────
    massflux_roe     = rho_roe * u_roe
    momentumflux_roe = rho_roe * u_roe * u_roe.T
    energyflux_roe   = rho_roe * H_roe * u_roe

    C = dot(massflux_roe, Aij)
    M = Matrix(momentumflux_roe.T @ Aij)
    G = p_roe * Aij
    K = dot(energyflux_roe, Aij)

    Ri = -Matrix([[C], M + G, [K]]) / Vi

    # ── Symbolic Jacobian ─────────────────────────────────────────────────────
    print("Computing Roe Jacobian symbolically (may take a while)...")
    all_vars    = Matrix([*Ui, *Uj])
    J_sym       = simplify(Ri.jacobian(all_vars))

    dRi_dUi_sym = J_sym[:, :5]
    dRi_dUj_sym = J_sym[:, 5:]

    # ── Lambdify ──────────────────────────────────────────────────────────────
    sym_args = (*Ui, *Uj, Ax, Ay, Az, Vi, g)

    dRi_dUi_fn = lambdify(sym_args, dRi_dUi_sym, modules="numpy")
    dRi_dUj_fn = lambdify(sym_args, dRi_dUj_sym, modules="numpy")

    def jacobian_fn(Ui_val, Uj_val, Aij_val, Vi_val, gamma_val):
        """
        Evaluate the Roe-averaged inviscid flux Jacobian numerically.

        Parameters
        ----------
        Ui_val    : array-like, shape (5,)  conservative vars of cell i  [rho, rho*ux, rho*uy, rho*uz, rho*E]
        Uj_val    : array-like, shape (5,)  conservative vars of cell j
        Aij_val   : array-like, shape (3,)  area-weighted face normal [Ax, Ay, Az]
        Vi_val    : float                   volume of cell i
        gamma_val : float                   ratio of specific heats

        Returns
        -------
        dRi_dUi : np.ndarray, shape (5, 5)   d(Ri)/d(Ui)
        dRi_dUj : np.ndarray, shape (5, 5)   d(Ri)/d(Uj)
        """
        import numpy as np
        Ui_val  = np.asarray(Ui_val,  dtype=float)
        Uj_val  = np.asarray(Uj_val,  dtype=float)
        Aij_val = np.asarray(Aij_val, dtype=float)

        args = (*Ui_val, *Uj_val, *Aij_val, float(Vi_val), float(gamma_val))

        dRi_dUi = np.array(dRi_dUi_fn(*args), dtype=float)
        dRi_dUj = np.array(dRi_dUj_fn(*args), dtype=float)

        return dRi_dUi, dRi_dUj

    return jacobian_fn


# ── Build once ────────────────────────────────────────────────────────────────
compute_inviscid_jacobian_roe = build_inviscid_jacobian_roe()


# ── Example usage ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import numpy as np

    gamma = 1.4
    rho_i, rho_j = 1.0, 1.1

    Ui = np.array([rho_i, rho_i*0.5, rho_i*0.2, 0.0, rho_i*2.5])
    Uj = np.array([rho_j, rho_j*0.4, rho_j*0.1, 0.0, rho_j*2.3])

    Aij = np.array([1.0, 0.0, 0.0])
    Vi  = 0.1

    dRi_dUi, dRi_dUj = compute_inviscid_jacobian_roe(Ui, Uj, Aij, Vi, gamma)

    np.set_printoptions(precision=4, suppress=True, linewidth=120)
    print("\ndRi/dUi (5x5):\n", dRi_dUi)
    print("\ndRi/dUj (5x5):\n", dRi_dUj)

    # ── Sanity check: columns should sum close to zero (flux consistency) ─────
    print("\nColumn sums of [dRi_dUi | dRi_dUj] (expect ~0 for uniform flow):")
    print(np.round(dRi_dUi.sum(axis=0), 6))
    print(np.round(dRi_dUj.sum(axis=0), 6))